In [2]:
!pip install onnx onnxruntime


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [1]:
import onnx
import onnxruntime as ort
import numpy as np

# Define input and output tensor names
input1_name = "X"
input2_name = "shape"

reshape_output_name = "Y"


# Create the ONNX model with Reshape operator
def create_reshape_model(input_rank, out_shape, dtype, allowzero=0):

    # Create "input-rank" input tensor
    input1 = onnx.helper.make_tensor_value_info(input1_name, dtype, input_rank)
    # Create "shape" input tensor (1D tensor of INT64)
    input2 = onnx.helper.make_tensor_value_info(input2_name, onnx.TensorProto.INT64, [None])

    # Create output tensor (final result after reshape operation)
    reshape_output = onnx.helper.make_tensor_value_info(reshape_output_name, dtype, out_shape)

    # Define reshape node
    reshape_node = onnx.helper.make_node(
        "Reshape",
        inputs=[input1_name, input2_name],
        allowzero=allowzero,
        outputs=[reshape_output_name],
    )

    # Create the ONNX graph (no output shape specified - will be inferred at runtime)
    graph_def = onnx.helper.make_graph(
        [reshape_node],
        "Reshape",
        [input1, input2],
        [reshape_output],
    )

    # Create the ONNX model
    model = onnx.helper.make_model(graph_def, opset_imports=[onnx.helper.make_opsetid("", 21)])
    model.ir_version = 9 
    onnx.checker.check_model(model)

    # Save the model
    onnx.save(model, "reshape.onnx")

    # Load and run the model with ONNX Runtime
    session = ort.InferenceSession("reshape.onnx")
    return session

def do_reshape(x, shape, session):
    # Run inference
    output = session.run(None, {input1_name: x, input2_name: shape})

    x_f = (np.array2string(x, separator=',', max_line_width=np.inf).replace('\n', ''))
    y_f = (np.array2string(output[0], separator=',', max_line_width=np.inf).replace('\n', ''))

    # Display results
    print(f"X shape: {x.shape}, X={x_f}")
    print(f"Output shape: {output[0].shape}, Result = {y_f}")


np.set_printoptions(precision=None, floatmode='fixed')

## Simple cases

In [2]:
# Case N1
# Input tensor: single rank tensor (int32)
# Shape tensor: [5] - 1D tensor with the length of the input tensor
onnx_type = onnx.TensorProto.INT32
x = np.array([-2, -1, 0, 1, 2], dtype=np.int32)
shape = [5]
session = create_reshape_model([None], shape, onnx_type)
shape = np.array(shape, dtype=np.int64)
do_reshape(x, shape, session)

X shape: (5,), X=[-2,-1, 0, 1, 2]
Output shape: (5,), Result = [-2,-1, 0, 1, 2]


In [5]:
# Case N2
# Input tensor: 2-rank tensor (int32)
# Shape tensor: [5, 2] - the shape is the inverted shape of the input tensor
onnx_type = onnx.TensorProto.INT32
x = np.array([[0, 1, 2, 3, 4],[5, 6, 7, 8, 9]], dtype=np.int32)
shape = [5, 2]
session = create_reshape_model([None, None], shape, onnx_type)
shape = np.array(shape, dtype=np.int64)
do_reshape(x, shape, session)

X shape: (2, 5), X=[[0,1,2,3,4], [5,6,7,8,9]]
Output shape: (5, 2), Result = [[0,1], [2,3], [4,5], [6,7], [8,9]]


In [6]:
# Case N3 - Test1
# Input tensor: 3-rank tensor (int32)
# Shape tensor: [10, 2]
onnx_type = onnx.TensorProto.INT32
x = np.array([[1, 2, 3, 4, 5], [6, 7, 8, 9, 10], [11, 12, 13, 14, 15], [16, 17, 18, 19, 20]], dtype=np.int32)
shape = [10, 2]
session = create_reshape_model([None, None], shape, onnx_type)
shape = np.array(shape, dtype=np.int64)
do_reshape(x, shape, session)

X shape: (4, 5), X=[[ 1, 2, 3, 4, 5], [ 6, 7, 8, 9,10], [11,12,13,14,15], [16,17,18,19,20]]
Output shape: (10, 2), Result = [[ 1, 2], [ 3, 4], [ 5, 6], [ 7, 8], [ 9,10], [11,12], [13,14], [15,16], [17,18], [19,20]]


In [7]:
# Case N3 - Test2
# Input tensor: 3-rank tensor (int32)
# Shape tensor: [5, 4]
onnx_type = onnx.TensorProto.INT32
x = np.array([[1, 2, 3, 4, 5], [6, 7, 8, 9, 10], [11, 12, 13, 14, 15], [16, 17, 18, 19, 20]], dtype=np.int32)
shape = [5, 2, 2]
session = create_reshape_model([None, None], shape, onnx_type)
shape = np.array(shape, dtype=np.int64)
do_reshape(x, shape, session)

X shape: (4, 5), X=[[ 1, 2, 3, 4, 5], [ 6, 7, 8, 9,10], [11,12,13,14,15], [16,17,18,19,20]]
Output shape: (5, 2, 2), Result = [[[ 1, 2],  [ 3, 4]], [[ 5, 6],  [ 7, 8]], [[ 9,10],  [11,12]], [[13,14],  [15,16]], [[17,18],  [19,20]]]


In [8]:
# Case N3 - Test3
# Input tensor: 2-rank tensor (int32)
# Shape tensor: [20, 1]
onnx_type = onnx.TensorProto.INT32
x = np.array([[1, 2, 3, 4, 5], [6, 7, 8, 9, 10], [11, 12, 13, 14, 15], [16, 17, 18, 19, 20]], dtype=np.int32)
shape = [20, 1]
session = create_reshape_model([None, None], shape, onnx_type)
shape = np.array(shape, dtype=np.int64)
do_reshape(x, shape, session)

X shape: (4, 5), X=[[ 1, 2, 3, 4, 5], [ 6, 7, 8, 9,10], [11,12,13,14,15], [16,17,18,19,20]]
Output shape: (20, 1), Result = [[ 1], [ 2], [ 3], [ 4], [ 5], [ 6], [ 7], [ 8], [ 9], [10], [11], [12], [13], [14], [15], [16], [17], [18], [19], [20]]


In [9]:
# This test MUST FAIL! (the shape does not match the number of elements)
# Case N3 - Test4
# Input tensor: 2-rank tensor (int32)
# Shape tensor: [6, 4] - Not compatinble (it would only be possible if we had 24 elements, we have 20)
onnx_type = onnx.TensorProto.INT32
x = np.array([[1, 2, 3, 4, 5], [6, 7, 8, 9, 10], [11, 12, 13, 14, 15], [16, 17, 18, 19, 20]], dtype=np.int32)
shape = [6, 4]
session = create_reshape_model([None, None], shape, onnx_type)
shape = np.array(shape, dtype=np.int64)
do_reshape(x, shape, session)

2026-03-09 10:51:37.974706536 [E:onnxruntime:, sequential_executor.cc:572 ExecuteKernel] Non-zero status code returned while running Reshape node. Name:'' Status Message: /onnxruntime_src/onnxruntime/core/providers/cpu/tensor/reshape_helper.h:45 onnxruntime::ReshapeHelper::ReshapeHelper(const onnxruntime::TensorShape&, onnxruntime::TensorShapeVector&, bool) input_shape_size == size was false. The input tensor cannot be reshaped to the requested shape. Input shape:{4,5}, requested shape:{6,4}



RuntimeException: [ONNXRuntimeError] : 6 : RUNTIME_EXCEPTION : Non-zero status code returned while running Reshape node. Name:'' Status Message: /onnxruntime_src/onnxruntime/core/providers/cpu/tensor/reshape_helper.h:45 onnxruntime::ReshapeHelper::ReshapeHelper(const onnxruntime::TensorShape&, onnxruntime::TensorShapeVector&, bool) input_shape_size == size was false. The input tensor cannot be reshaped to the requested shape. Input shape:{4,5}, requested shape:{6,4}


## Using (-1) to autoinfer one dimension

In [10]:
# Case N1
# Input tensor: single rank tensor (int32)
# Shape tensor: [-1] -- coverts to --> [5] (Auto infered dimension)
onnx_type = onnx.TensorProto.INT32
x = np.array([-2, -1, 0, 1, 2], dtype=np.int32)
shape = [-1]
session = create_reshape_model([None], shape, onnx_type)
shape = np.array(shape, dtype=np.int64)
do_reshape(x, shape, session)

X shape: (5,), X=[-2,-1, 0, 1, 2]
Output shape: (5,), Result = [-2,-1, 0, 1, 2]


In [11]:
# Case N2
# Input tensor: 2-rank tensor (int32)
# Shape tensor: [5, -1] -- coverts to --> [5, 2] (Auto infered dimension)
onnx_type = onnx.TensorProto.INT32
x = np.array([[-20,-10,0,10,20],[-15,-10,5,10,0]], dtype=np.int32)
shape = [5, -1]
session = create_reshape_model([None, None], shape, onnx_type)
shape = np.array(shape, dtype=np.int64)
do_reshape(x, shape, session)

X shape: (2, 5), X=[[-20,-10,  0, 10, 20], [-15,-10,  5, 10,  0]]
Output shape: (5, 2), Result = [[-20,-10], [  0, 10], [ 20,-15], [-10,  5], [ 10,  0]]


In [12]:
# Case N3 - Test1
# Input tensor: 2-rank tensor (int32)
# Shape tensor: [10, -1] -- coverts to --> [10, 2] (Auto infered dimension)
onnx_type = onnx.TensorProto.INT32
x = np.array([[1, 2, 3, 4, 5], [6, 7, 8, 9, 10], [11, 12, 13, 14, 15], [16, 17, 18, 19, 20]], dtype=np.int32)
shape = [10, -1]
session = create_reshape_model([None, None], shape, onnx_type)
shape = np.array(shape, dtype=np.int64)
do_reshape(x, shape, session)

X shape: (4, 5), X=[[ 1, 2, 3, 4, 5], [ 6, 7, 8, 9,10], [11,12,13,14,15], [16,17,18,19,20]]
Output shape: (10, 2), Result = [[ 1, 2], [ 3, 4], [ 5, 6], [ 7, 8], [ 9,10], [11,12], [13,14], [15,16], [17,18], [19,20]]


In [13]:
# This test MUST FAIL! (only one dimension can be auto inferred)
# Case N3 - Test2
# Input tensor: 2-rank tensor (int32)
# Shape tensor: [-1, -1] --> conversion not allowed
onnx_type = onnx.TensorProto.INT32
x = np.array([[1, 2, 3, 4, 5], [6, 7, 8, 9, 10], [11, 12, 13, 14, 15], [16, 17, 18, 19, 20]], dtype=np.int32)
shape = [-1, -1]
session = create_reshape_model([None, None], shape, onnx_type)
shape = np.array(shape, dtype=np.int64)
do_reshape(x, shape, session)

2026-03-09 10:51:38.951117749 [E:onnxruntime:, sequential_executor.cc:572 ExecuteKernel] Non-zero status code returned while running Reshape node. Name:'' Status Message: /onnxruntime_src/onnxruntime/core/providers/cpu/tensor/reshape_helper.h:24 onnxruntime::ReshapeHelper::ReshapeHelper(const onnxruntime::TensorShape&, onnxruntime::TensorShapeVector&, bool) unknown_dim == -1 was false. At most one dimension can be -1.



RuntimeException: [ONNXRuntimeError] : 6 : RUNTIME_EXCEPTION : Non-zero status code returned while running Reshape node. Name:'' Status Message: /onnxruntime_src/onnxruntime/core/providers/cpu/tensor/reshape_helper.h:24 onnxruntime::ReshapeHelper::ReshapeHelper(const onnxruntime::TensorShape&, onnxruntime::TensorShapeVector&, bool) unknown_dim == -1 was false. At most one dimension can be -1.


In [14]:
# Case N4
# Input tensor: 3-rank tensor (int32)
# Shape tensor: [0, 6, -1] -- coverts to --> [2, 6, 4] (Auto infered dimension)
onnx_type = onnx.TensorProto.INT32
x = np.arange(24, dtype=np.int32).reshape(2, 3, 2, 2)
shape = [0, 6, -1]
session = create_reshape_model([None, None, None, None], [2, 6, 2], onnx_type, allowzero=0)
shape = np.array(shape, dtype=np.int64)
do_reshape(x, shape, session)

X shape: (2, 3, 2, 2), X=[[[[ 0, 1],   [ 2, 3]],  [[ 4, 5],   [ 6, 7]],  [[ 8, 9],   [10,11]]], [[[12,13],   [14,15]],  [[16,17],   [18,19]],  [[20,21],   [22,23]]]]
Output shape: (2, 6, 2), Result = [[[ 0, 1],  [ 2, 3],  [ 4, 5],  [ 6, 7],  [ 8, 9],  [10,11]], [[12,13],  [14,15],  [16,17],  [18,19],  [20,21],  [22,23]]]


## Using **allowzero**

In this section the shape will be explicitly passed to the `create_reshape_model`.

However, `do_reshape` is taking into account the intended shape.

In [15]:
# Case N1: Test1
# Input tensor: single rank tensor (int32)
# Allowzero: If not specified its value is 0
# Shape tensor: [0] -- converts to --> [5] (auto infered dimension from the input tensor)
onnx_type = onnx.TensorProto.INT32
x = np.array([-2, -1, 0, 1, 2], dtype=np.int32)
session = create_reshape_model([None], [5], onnx_type)
shape = [0]
shape = np.array(shape, dtype=np.int64)
do_reshape(x, shape, session)

X shape: (5,), X=[-2,-1, 0, 1, 2]
Output shape: (5,), Result = [-2,-1, 0, 1, 2]


In [16]:
# This cell MUST FAIL!
# Case N1: Test2
# Input tensor: single rank tensor (int32)
# Allowzero: 1
# Shape tensor: [0] (it is not possible to infer the dimension when allowzero is not 0)
onnx_type = onnx.TensorProto.INT32
x = np.array([-2, -1, 0, 1, 2], dtype=np.int32)
session = create_reshape_model([None], [5], onnx_type, allowzero=1)
shape = [5,0]
shape = np.array(shape, dtype=np.int64)
do_reshape(x, shape, session)

2026-03-09 10:51:39.551449155 [E:onnxruntime:, sequential_executor.cc:572 ExecuteKernel] Non-zero status code returned while running Reshape node. Name:'' Status Message: /onnxruntime_src/onnxruntime/core/providers/cpu/tensor/reshape_helper.h:45 onnxruntime::ReshapeHelper::ReshapeHelper(const onnxruntime::TensorShape&, onnxruntime::TensorShapeVector&, bool) input_shape_size == size was false. The input tensor cannot be reshaped to the requested shape. Input shape:{5}, requested shape:{5,0}



RuntimeException: [ONNXRuntimeError] : 6 : RUNTIME_EXCEPTION : Non-zero status code returned while running Reshape node. Name:'' Status Message: /onnxruntime_src/onnxruntime/core/providers/cpu/tensor/reshape_helper.h:45 onnxruntime::ReshapeHelper::ReshapeHelper(const onnxruntime::TensorShape&, onnxruntime::TensorShapeVector&, bool) input_shape_size == size was false. The input tensor cannot be reshaped to the requested shape. Input shape:{5}, requested shape:{5,0}


In [17]:
# Case N2
# Input tensor: 3-rank tensor (int32)
# Allowzero: If not specified its value is 0
# Shape tensor: [0, 0] -- converts to --> [4, 5] (auto infered dimension from the input tensor)
onnx_type = onnx.TensorProto.INT32
x = np.array([[1, 2, 3, 4, 5], [6, 7, 8, 9, 10], [11, 12, 13, 14, 15], [16, 17, 18, 19, 20]], dtype=np.int32)
session = create_reshape_model([None, None], [4, 5], onnx_type)
shape = [0, 0]
shape = np.array(shape, dtype=np.int64)
do_reshape(x, shape, session)

X shape: (4, 5), X=[[ 1, 2, 3, 4, 5], [ 6, 7, 8, 9,10], [11,12,13,14,15], [16,17,18,19,20]]
Output shape: (4, 5), Result = [[ 1, 2, 3, 4, 5], [ 6, 7, 8, 9,10], [11,12,13,14,15], [16,17,18,19,20]]


In [18]:
# Case N3
# Input tensor: 3-rank tensor (int32)
# Allowzero: If not specified its value is 0
# Shape tensor: [0, -1] -- converts to --> [4, 5]
# (0 is infered from input tensor)
# (-1 is auto infered based on the other dimensions)
onnx_type = onnx.TensorProto.INT32
x = np.array([[0, 1, 2, 3, 4], [5, 6, 7, 8, 9]], dtype=np.int32)
session = create_reshape_model([None, None], [4, 5], onnx_type)
shape = [0, -1]
shape = np.array(shape, dtype=np.int64)
do_reshape(x, shape, session)

X shape: (2, 5), X=[[0,1,2,3,4], [5,6,7,8,9]]
Output shape: (2, 5), Result = [[0,1,2,3,4], [5,6,7,8,9]]


2026-03-09 10:51:39.872832468 [W:onnxruntime:, execution_frame.cc:876 VerifyOutputSizes] Expected shape from model of {4,5} does not match actual shape of {2,5} for output Y


In [19]:
# Case N4
# Input tensor: 3-rank tensor (int32)
# Shape tensor: [0, -1] -- converts to --> [4, 5]
# (0 is infered from input tensor)
# (-1 is auto infered based on the other dimensions)
onnx_type = onnx.TensorProto.INT32
x = np.array([], dtype=np.int32).reshape(0, 3, 8)
shape = [1, 2, 0]
session = create_reshape_model([None, None, None], shape, onnx_type, allowzero=1)
shape = np.array(shape, dtype=np.int64)
do_reshape(x, shape, session)

X shape: (0, 3, 8), X=[]
Output shape: (1, 2, 0), Result = []


## Using empty shape (converting to scalar)

In [20]:
# Case N1
# Input tensor: 2-rank tensor (int32)
# Allowzero: If not specified its value is 0
# Shape tensor: [] -- converts to --> scalar
onnx_type = onnx.TensorProto.INT32
x = np.array([[1]], dtype=np.int32)
shape = []
session = create_reshape_model([None, None], shape, onnx_type)
shape = np.array(shape, dtype=np.int64)
do_reshape(x, shape, session)

X shape: (1, 1), X=[[1]]
Output shape: (), Result = 1


In [21]:
# Case N2
# Input tensor: 4-rank tensor (int32) with 1 element
# Shape tensor: [] -- converts to --> scalar
onnx_type = onnx.TensorProto.INT32
x = np.array([[[[1]]]], dtype=np.int32)
shape = []
session = create_reshape_model([None, None, None, None], shape, onnx_type)
shape = np.array([], dtype=np.int64)
do_reshape(x, shape, session)

X shape: (1, 1, 1, 1), X=[[[[1]]]]
Output shape: (), Result = 1


In [22]:
# This cell MUST FAIL!
# Case N3 - Empty shape test with float32 (converts to scalar)
# Input tensor: 1-element tensor (float32)
# Shape tensor: [] (empty) -- converts to --> scalar (rank 0)
onnx_type = onnx.TensorProto.FLOAT
x = np.array([1, 2], dtype=np.float32)
shape = []
session = create_reshape_model([None, None], shape, onnx_type)
shape = np.array(shape, dtype=np.int64)
do_reshape(x, shape, session)

InvalidArgument: [ONNXRuntimeError] : 2 : INVALID_ARGUMENT : Invalid rank for input: X Got: 1 Expected: 2 Please fix either the inputs/outputs or the model.

In [23]:
# This cell MUST FAIL!
# Case N1: Test2
# Input tensor: single rank tensor (int32)
# Allowzero: 1
# Shape tensor: [0] (it is not possible to infer the dimension when allowzero is not 0)
onnx_type = onnx.TensorProto.INT32
x = np.zeros((2, 4, 0, 4, 2, 5, 2, 4, 5), dtype=np.float32)  # shape: (2, 0, 3)
shape = [0, 0, 0, 0, 0]  # Mantém as dimensões zero
session = create_reshape_model([None, None, None, None, None, None, None, None, None], shape, onnx.TensorProto.FLOAT, allowzero=1)
shape = np.array(shape, dtype=np.int64)
do_reshape(x, shape, session)

X shape: (2, 4, 0, 4, 2, 5, 2, 4, 5), X=[]
Output shape: (0, 0, 0, 0, 0), Result = []


In [24]:
# This cell MUST FAIL!
# Case N1: Test2
# Input tensor: single rank tensor (int32)
# Allowzero: 1
# Shape tensor: [0] (it is not possible to infer the dimension when allowzero is not 0)
onnx_type = onnx.TensorProto.INT32
x = np.zeros((2, 4, 0), dtype=np.float32)  # shape: (2, 0, 3)
shape = [0,0,0]  # Mantém as dimensões zero
session = create_reshape_model([None, None, None], shape, onnx.TensorProto.FLOAT, allowzero=0)
shape = np.array(shape, dtype=np.int64)
do_reshape(x, shape, session)

X shape: (2, 4, 0), X=[]
Output shape: (2, 4, 0), Result = []


2026-03-09 10:51:42.603432319 [W:onnxruntime:, execution_frame.cc:876 VerifyOutputSizes] Expected shape from model of {0,0,0} does not match actual shape of {2,4,0} for output Y


In [25]:
# Case N1
# Input tensor: single rank tensor (int32)
# Shape tensor: [5] - 1D tensor with the length of the input tensor
onnx_type = onnx.TensorProto.INT32
x = np.arange(100, dtype=np.int32).reshape(25, 1, 4, 1)
shape = [2, -1, 0]
session = create_reshape_model([None, None, None, None], shape, onnx_type, allowzero = 1)
shape = np.array(shape, dtype=np.int64)
do_reshape(x, shape, session)

2026-03-09 10:51:42.919452225 [E:onnxruntime:, sequential_executor.cc:572 ExecuteKernel] Non-zero status code returned while running Reshape node. Name:'' Status Message: /onnxruntime_src/onnxruntime/core/providers/cpu/tensor/reshape_helper.h:39 onnxruntime::ReshapeHelper::ReshapeHelper(const onnxruntime::TensorShape&, onnxruntime::TensorShapeVector&, bool) size != 0 && (input_shape_size % size) == 0 was false. The input tensor cannot be reshaped to the requested shape. Input shape:{25,1,4,1}, requested shape:{2,-1,0}



RuntimeException: [ONNXRuntimeError] : 6 : RUNTIME_EXCEPTION : Non-zero status code returned while running Reshape node. Name:'' Status Message: /onnxruntime_src/onnxruntime/core/providers/cpu/tensor/reshape_helper.h:39 onnxruntime::ReshapeHelper::ReshapeHelper(const onnxruntime::TensorShape&, onnxruntime::TensorShapeVector&, bool) size != 0 && (input_shape_size % size) == 0 was false. The input tensor cannot be reshaped to the requested shape. Input shape:{25,1,4,1}, requested shape:{2,-1,0}


In [26]:
# Case N4
# Input tensor: 3-rank tensor (int32)
# Shape tensor: [0, -1] -- converts to --> [4, 5]
# (0 is infered from input tensor)
# (-1 is auto infered based on the other dimensions)
onnx_type = onnx.TensorProto.INT32
x = np.array([], dtype=np.int32).reshape(0, 3, 8)
shape = [1, 2, -1]
session = create_reshape_model([None, None, None], shape, onnx_type, allowzero=1)
shape = np.array(shape, dtype=np.int64)
do_reshape(x, shape, session)

X shape: (0, 3, 8), X=[]
Output shape: (1, 2, 0), Result = []
